## Instalação e Montagem do Drive


In [1]:
# Célula 1: Instalação
!pip install pyspark delta-spark -q

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.1 MB/s eta 0:00:00a 0:00:01
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Inicialização do PySpark com otimizações

In [2]:
import pyspark
from delta import *


builder = pyspark.sql.SparkSession.builder \
    .appName("GeoBusiness_Analytics_BR_Bronze_Delta") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.driver.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "10")

# Instanciamos o Spark injetando as dependências do Delta
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark rodando com Delta Lake! Versão:", spark.version)

Spark rodando com Delta Lake! Versão: 4.0.2


In [3]:
from datetime import datetime

In [6]:
lakeBasePath = '/content/drive/MyDrive/Datalake'


landingBasePath = f'{lakeBasePath}/landing-lake/receita'
bronzeBasePath = f'{lakeBasePath}/bronze-lake/receita'

In [7]:
def load_raw_layer(tableName, fields, encoding = 'ISO-8859-1' , dataSource = ''):
    if dataSource == 'csv':

        start_time = datetime.now()
        print(f'Iniciando carga da tabela {tableName}')
        try:
            landingPath = f'{landingBasePath}/{tableName}'
            bronzePath = f'{bronzeBasePath}/{tableName}'
            tableColumns = fields
            tempDf = (
                spark
                .read
                .option("delimiter", ";")
                .option("encoding", encoding)
                .csv(f"{landingPath}/*.csv", inferSchema=True, header=False)
            ).toDF(*tableColumns)

            (
                tempDf
                .write
                .format("delta")
                .mode("overwrite")
                .save(bronzePath)
            )

            status = 'Sucesso'

        except Exception as e:
            print(f'Erro no processamento da tabela \n{e}')
            status = 'Erro'

        
        end_time = datetime.now()
        duration = end_time - start_time

        print(f"[{end_time.strftime('%Y-%m-%d %H:%M:%S')}] Finalizada carga da tabela: {tableName} | Status: {status} | Duração: {duration}")
        return        

In [8]:
tables = {
    "cnaes": [
        "codigo", "descricao"
    ], 
    
    "empresas": [
        "cnpj_basico", "razao_social", "natureza_juridica", 
        "qualificacao_responsavel", "capital_social", "porte_empresa", 
        "ente_federativo_responsavel"
    ], 
    
    "estabelecimentos": [
        "cnpj_basico", "cnpj_ordem", "cnpj_dv", "identificador_matriz_filial", 
        "nome_fantasia", "situacao_cadastral", "data_situacao_cadastral", 
        "motivo_situacao_cadastral", "nome_cidade_exterior", "pais", 
        "data_inicio_atividade", "cnae_fiscal_principal", "cnae_fiscal_secundaria", 
        "tipo_logradouro", "logradouro", "numero", "complemento", "bairro", 
        "cep", "uf", "municipio", "ddd_1", "telefone_1", "ddd_2", "telefone_2", 
        "ddd_fax", "fax", "correio_eletronico", "situacao_especial", "data_situacao_especial"
    ], 
    
    "municipios": [
        "codigo", "descricao"
    ], 
    
    "naturezas": [
        "codigo", "descricao"
    ], 
    
    "paises": [
        "codigo", "descricao"
    ], 
    
    "qualificacoes": [
        "codigo", "descricao"
    ], 
    
    "simples": [
        "cnpj_basico", "opcao_simples", "data_opcao_simples", "data_exclusao_simples", 
        "opcao_mei", "data_opcao_mei", "data_exclusao_mei"
    ], 
    
    "socios": [
        "cnpj_basico", "identificador_socio", "nome_socio_razao_social", 
        "cnpj_cpf_socio", "qualificacao_socio", "data_entrada_sociedade", 
        "pais", "representante_legal", "nome_representante", 
        "qualificacao_representante_legal", "faixa_etaria"
    ] 
}

In [9]:
for tableName, fields in tables.items():
    load_raw_layer(tableName, fields,  dataSource = 'csv')

Iniciando carga da tabela cnaes
[2026-05-22 02:47:59] Finalizada carga da tabela: cnaes | Status: Sucesso | Duração: 0:00:15.611817
Iniciando carga da tabela empresas
[2026-05-22 02:52:05] Finalizada carga da tabela: empresas | Status: Sucesso | Duração: 0:04:06.672497
Iniciando carga da tabela estabelecimentos
[2026-05-22 03:06:10] Finalizada carga da tabela: estabelecimentos | Status: Sucesso | Duração: 0:14:04.881152
Iniciando carga da tabela municipios
[2026-05-22 03:06:12] Finalizada carga da tabela: municipios | Status: Sucesso | Duração: 0:00:01.749330
Iniciando carga da tabela naturezas
[2026-05-22 03:06:14] Finalizada carga da tabela: naturezas | Status: Sucesso | Duração: 0:00:01.733479
Iniciando carga da tabela paises
[2026-05-22 03:06:16] Finalizada carga da tabela: paises | Status: Sucesso | Duração: 0:00:02.297350
Iniciando carga da tabela qualificacoes
[2026-05-22 03:06:19] Finalizada carga da tabela: qualificacoes | Status: Sucesso | Duração: 0:00:02.758829
Iniciando ca

In [ ]:
for a,b in tables.items():
    print(a,b)

cnaes ['codigo', 'descricao']
empresas ['cnpj_basico', 'razao_social', 'natureza_juridica', 'qualificacao_responsavel', 'capital_social', 'porte_empresa', 'ente_federativo_responsavel']
estabelecimentos ['cnpj_basico', 'cnpj_ordem', 'cnpj_dv', 'identificador_matriz_filial', 'nome_fantasia', 'situacao_cadastral', 'data_situacao_cadastral', 'motivo_situacao_cadastral', 'nome_cidade_exterior', 'pais', 'data_inicio_atividade', 'cnae_fiscal_principal', 'cnae_fiscal_secundaria', 'tipo_logradouro', 'logradouro', 'numero', 'complemento', 'bairro', 'cep', 'uf', 'municipio', 'ddd_1', 'telefone_1', 'ddd_2', 'telefone_2', 'ddd_fax', 'fax', 'correio_eletronico', 'situacao_especial', 'data_situacao_especial']
municipios ['codigo', 'descricao']
naturezas ['codigo', 'descricao']
paises ['codigo', 'descricao']
qualificacoes ['codigo', 'descricao']
simples ['cnpj_basico', 'opcao_simples', 'data_opcao_simples', 'data_exclusao_simples', 'opcao_mei', 'data_opcao_mei', 'data_exclusao_mei']
socios ['cnpj_bas

In [ ]:
spark.stop()

NameError: name 'spark' is not defined